# 01 · Fuentes: de dónde salen los datos y cómo se unen

*Backblaze publica cada trimestre el estado diario de unos 340.000 discos de sus centros de datos. Elijo de sus zips 9 fechas de referencia, una por semana entre el 1 de enero y el 26 de febrero de 2026. Para cada disco guardo sus lecturas de ese día y las de 7, 14 y 30 días antes. Diciembre de 2025 aporta el historial para las primeras fechas; no se usa para entrenar.*

*Al final, todo se une por número de serie y se crea el target: `falla_30d`.*

## Preparación: leer los zips sin descomprimirlos

*Cada zip pesa más de 1 GB y descomprimido ocuparía unos 25. Con `zipfile` leo cada día directamente desde dentro, y solo las columnas que hacen falta. Las constantes en mayúsculas son los parámetros del proyecto: si mañana quisiera 30.000 filas, bastaría con cambiar `TOTAL_FILAS`.*

In [1]:
import os
import zipfile

import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

CARPETA_ZIPS = "C:/Users/sagsh/Downloads/"          # los zips se quedan fuera del repositorio
ZIPS = ["data_Q4_2025.zip", "data_Q1_2026.zip"]      # diciembre de 2025 solo se usa para el historial
FOTOS = [str((pd.Timestamp("2026-01-01") + pd.Timedelta(days = 7 * k)).date()) for k in range(9)]
HORIZONTE = 30
FIN_Q1 = "2026-03-31"
DIAS_ANTES = [7, 14, 30]
TOTAL_FILAS = 26000
SMART = [1, 3, 4, 5, 7, 9, 10, 12, 183, 184, 187, 188, 189, 190, 192, 193, 194, 197, 198, 199, 240, 241, 242]

columnas = ["date", "serial_number", "model", "capacity_bytes", "datacenter"]
for n in SMART:
    columnas += ["smart_" + str(n) + "_normalized", "smart_" + str(n) + "_raw"]
columnas_antes = ["serial_number"] + ["smart_" + str(n) + "_raw" for n in SMART]

# Índice: en qué zip y con qué nombre está el archivo de cada día
dias = {}
for z in ZIPS:
    with zipfile.ZipFile(CARPETA_ZIPS + z) as zf:
        for nombre in zf.namelist():
            base = os.path.basename(nombre)
            if base.endswith(".csv") and not base.startswith("._"):
                dias[base.replace(".csv", "")] = (CARPETA_ZIPS + z, nombre)

def leer_dia(fecha, cols):
    ruta, nombre = dias[fecha]
    with zipfile.ZipFile(ruta) as zf:
        cabecera = pd.read_csv(zf.open(nombre), nrows = 0).columns
        return pd.read_csv(zf.open(nombre), usecols = [c for c in cols if c in cabecera], low_memory = False)

print("Días disponibles:", len(dias), "| de", min(dias), "a", max(dias))

Días disponibles: 182 | de 2025-10-01 a 2026-03-31


## Fuente 1: los fallos del primer trimestre

*Un disco "falla" el último día que funcionó: ese día lleva `failure = 1`. Recorro los 90 archivos de enero a marzo y me quedo con esas filas.*

*Hay una comprobación rápida. Backblaze publica que en el trimestre fallaron 1.030 discos, así que si mi tabla tiene 1.030 filas, la extracción está bien hecha.*

In [2]:
fechas_q1 = [f for f in sorted(dias) if "2026-01-01" <= f <= FIN_Q1]
fallos = []
for fecha in fechas_q1:
    d = leer_dia(fecha, ["date", "serial_number", "failure"])
    fallos.append(d[d["failure"] == 1][["serial_number", "date"]])
fallos = pd.concat(fallos).drop_duplicates("serial_number")
fallos.to_csv("data/raw/fallos_q1_2026.csv", index = False)
fecha_fallo = pd.to_datetime(fallos.set_index("serial_number")["date"])
print("Días leídos:", len(fechas_q1), "| fallos:", len(fallos))

Días leídos: 90 | fallos: 1030


## Fuente 2: la foto de cada fecha de referencia

*En cada una de las 9 fechas leo el estado de toda la flota, unos 340.000 discos. Me quedo con todos los que fallan en los 30 días siguientes, que son pocos y muy valiosos, y con una muestra al azar de sanos hasta llegar a 26.000 filas. Quito los discos que ya habían fallado antes de la fecha: ya no están en la flota.*

*Un mismo disco que va a fallar aparece en varias fechas, cada vez más cerca del fallo. Eso le enseña al modelo cómo empeora.*

In [3]:
positivos, sanos, resumen = [], [], []
for foto in FOTOS:
    hoy = leer_dia(foto, columnas)
    f = hoy["serial_number"].map(fecha_fallo)
    hoy = hoy[~(f <= pd.Timestamp(foto))]                   # fuera los que ya habían fallado
    f = hoy["serial_number"].map(fecha_fallo)
    limite = min(pd.Timestamp(foto) + pd.Timedelta(days = HORIZONTE), pd.Timestamp(FIN_Q1))
    falla = (f > pd.Timestamp(foto)) & (f <= limite)
    rng = np.random.default_rng(42 + FOTOS.index(foto))
    positivos.append(hoy[falla].assign(fecha_foto = foto))
    sanos.append(hoy[~falla].iloc[rng.choice((~falla).sum(), size = 3500, replace = False)].assign(fecha_foto = foto))
    resumen.append([foto, len(hoy), int(falla.sum())])
    print(foto, "| activos:", len(hoy), "| fallan en 30 días:", int(falla.sum()))

positivos = pd.concat(positivos)
sanos_por_foto = (TOTAL_FILAS - len(positivos)) // len(FOTOS)
muestra = pd.concat([positivos] + [s.head(sanos_por_foto) for s in sanos])
muestra.to_csv("data/raw/fotos_hoy_q1_2026.csv", index = False)
resumen = pd.DataFrame(resumen, columns = ["fecha_foto", "discos_activos", "fallan_30d"]).assign(sanos_muestra = sanos_por_foto)
resumen.to_csv("data/raw/resumen_extraccion.csv", index = False)
print("Filas:", len(muestra), "| filas que fallan:", len(positivos), "| discos que fallan:", positivos["serial_number"].nunique())

2026-01-01 | activos: 338759 | fallan en 30 días: 264
2026-01-08 | activos: 338776 | fallan en 30 días: 287
2026-01-15 | activos: 336679 | fallan en 30 días: 328
2026-01-22 | activos: 336845 | fallan en 30 días: 311
2026-01-29 | activos: 337879 | fallan en 30 días: 300
2026-02-05 | activos: 337495 | fallan en 30 días: 294
2026-02-12 | activos: 338711 | fallan en 30 días: 336
2026-02-19 | activos: 339157 | fallan en 30 días: 320
2026-02-26 | activos: 340417 | fallan en 30 días: 324
Filas: 25993 | filas que fallan: 2764 | discos que fallan: 902


## Fuente 3: el historial de cada disco

*Una foto sola no dice si un disco empeora. Por eso, para cada disco de la muestra, leo sus valores SMART de 7, 14 y 30 días antes de su fecha de referencia. En la fase de limpieza los convertiré en tendencias: cuántos sectores dañados nuevos han aparecido en la última semana, en dos semanas y en un mes.*

*Aquí es donde entra diciembre de 2025: la fecha del 1 de enero necesita las lecturas del 2, el 18 y el 25 de diciembre.*

In [4]:
for d in DIAS_ANTES:
    trozos = []
    for foto in FOTOS:
        dia = str((pd.Timestamp(foto) - pd.Timedelta(days = d)).date())
        lectura = leer_dia(dia, columnas_antes)
        discos = muestra.loc[muestra["fecha_foto"] == foto, "serial_number"]
        trozos.append(lectura[lectura["serial_number"].isin(discos)].assign(fecha_foto = foto))
    antes = pd.concat(trozos)
    antes.to_csv("data/raw/fotos_hace" + str(d) + "_q1_2026.csv", index = False)
    print("Lecturas de hace", d, "días:", antes.shape)

Lecturas de hace 7 días: (25833, 25)
Lecturas de hace 14 días: (25757, 25)
Lecturas de hace 30 días: (25697, 25)


## Prueba final: la foto del 1 de abril

*Guardo aparte una foto de un mes que el modelo nunca verá mientras lo construyo: el 1 de abril de 2026, con los fallos hasta el 1 de mayo. Quito cualquier disco que ya esté en mi muestra de enero a marzo, para que sean discos nuevos de verdad. Mantengo la misma proporción de fallos que en mi muestra. Al final del notebook 03 será la prueba de fuego: si el modelo acierta aquí, generaliza al futuro.*

In [ ]:
# Añado el zip de Q2 al índice de días
with zipfile.ZipFile(CARPETA_ZIPS + "data_Q2_2026.zip") as zf:
    for nombre in zf.namelist():
        base = os.path.basename(nombre)
        if base.endswith(".csv") and not base.startswith("._"):
            dias[base.replace(".csv", "")] = (CARPETA_ZIPS + "data_Q2_2026.zip", nombre)

FOTO_ABRIL = "2026-04-01"
limite = pd.Timestamp(FOTO_ABRIL) + pd.Timedelta(days = HORIZONTE)

# Fallos de abril (del 1 de abril al 1 de mayo)
fechas_abril = [f for f in sorted(dias) if FOTO_ABRIL <= f <= str(limite.date())]
fa = []
for fecha in fechas_abril:
    d = leer_dia(fecha, ["date", "serial_number", "failure"])
    fa.append(d[d["failure"] == 1][["serial_number", "date"]])
fallos_abril = pd.concat(fa).drop_duplicates("serial_number")
fallos_abril.to_csv("data/raw/fallos_abril_2026.csv", index = False)
fecha_fallo_todo = pd.to_datetime(pd.concat([fallos, fallos_abril]).drop_duplicates("serial_number").set_index("serial_number")["date"])

# Foto del 1 de abril: sin discos de la muestra de Q1 y con la misma proporción de fallos
hoy = leer_dia(FOTO_ABRIL, columnas)
f = hoy["serial_number"].map(fecha_fallo_todo)
hoy = hoy[~(f <= pd.Timestamp(FOTO_ABRIL)) & ~hoy["serial_number"].isin(muestra["serial_number"])]
f = hoy["serial_number"].map(fecha_fallo_todo)
falla = (f > pd.Timestamp(FOTO_ABRIL)) & (f <= limite)
proporcion = len(positivos) / len(muestra)
n_sanos = int(round(falla.sum() * (1 - proporcion) / proporcion))
rng = np.random.default_rng(42)
sanos_abril = hoy[~falla].iloc[rng.choice((~falla).sum(), size = n_sanos, replace = False)]
abril = pd.concat([hoy[falla], sanos_abril]).assign(fecha_foto = FOTO_ABRIL)
abril.to_csv("data/raw/fotos_hoy_abril_2026.csv", index = False)
print("Días de fallos leídos:", len(fechas_abril), "| fallos en abril:", len(fallos_abril))
print("Abril | discos:", len(abril), "| fallan en 30 días:", int(falla.sum()), "| sanos:", len(sanos_abril))

# Historial: lecturas del 25, 18 y 2 de marzo (7, 14 y 30 días antes), que están en el zip de Q1
for d in DIAS_ANTES:
    dia = str((pd.Timestamp(FOTO_ABRIL) - pd.Timedelta(days = d)).date())
    lectura = leer_dia(dia, columnas_antes)
    antes = lectura[lectura["serial_number"].isin(abril["serial_number"])].assign(fecha_foto = FOTO_ABRIL)
    antes.to_csv("data/raw/fotos_hace" + str(d) + "_abril_2026.csv", index = False)
    print("Abril, hace", d, "días:", antes.shape)

Días de fallos leídos: 31 | fallos en abril: 691
Abril | discos: 4411 | fallan en 30 días: 469 | sanos: 3942
Abril, hace 7 días: (4318, 25)
Abril, hace 14 días: (4337, 25)
Abril, hace 30 días: (4264, 25)
